    # Week 3: NumPy MLP and the Adam optimizer

    [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuranLearning/ai-llm-deep-dive/blob/main/notebooks/week-03-numpy-mlp-adam.ipynb)
    · [Lesson text](https://github.com/QuranLearning/ai-llm-deep-dive/blob/main/lessons/week-03.md) · [Learning Hub week page](https://claude.ai/artifact/VwTfCas4FMnB23QHuBUJhj#w-ai-llm-3)

    **By the end of this notebook you will:**
    - train logistic regression and see where a straight line fails
- train a 2-layer MLP in pure NumPy on two spirals
- implement Adam and compare it with plain SGD

    **How to use it:** work top to bottom. Each exercise has a starter cell (replace the `TODO`s) and a **Check** cell that tells you if you got it right.
    Stuck for more than 15 minutes? Ask the tutor on the week page, then peek at the **Solutions** section at the bottom.
    In Colab, use *File → Save a copy in Drive* so your work is saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
def ok(msg="Correct"): print("\u2705", msg)
def sigmoid(z): return 1 / (1 + np.exp(-z))
def accuracy(params, X, y): return np.mean(forward(params, X)[1].argmax(1) == y)   # uses your forward() from Exercise 2

def make_spirals(n=300, noise=0.2, seed=0):
    r = np.random.default_rng(seed)
    t = np.sqrt(r.uniform(0, 1, n)) * 3 * np.pi
    a = np.c_[t * np.cos(t), t * np.sin(t)]
    X = np.r_[a, -a] / 10 + r.normal(scale=noise / 10 * 3, size=(2 * n, 2))
    y = np.r_[np.zeros(n, int), np.ones(n, int)]
    return X, y

def make_blobs(n=200, seed=0):
    r = np.random.default_rng(seed)
    X = np.r_[r.normal([-1, -1], 0.6, (n, 2)), r.normal([1, 1], 0.6, (n, 2))]
    return X, np.r_[np.zeros(n, int), np.ones(n, int)]

def plot_boundary(predict, X, y, title=""):
    xx, yy = np.meshgrid(np.linspace(X[:,0].min()-.3, X[:,0].max()+.3, 200), np.linspace(X[:,1].min()-.3, X[:,1].max()+.3, 200))
    Z = predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, Z, alpha=.25, cmap="coolwarm"); plt.scatter(X[:,0], X[:,1], c=y, s=8, cmap="coolwarm"); plt.title(title); plt.show()

Xs, ys = make_spirals(); Xb, yb = make_blobs()
plt.scatter(Xs[:,0], Xs[:,1], c=ys, s=8, cmap="coolwarm"); plt.title("two spirals"); plt.show()

## Exercise 1: logistic regression
Write `logreg_loss_grads(w, b, X, y)` returning `(L, dw, db)` (binary cross-entropy, mean over the batch), then `train_logreg(X, y, lr, steps)` with plain gradient descent, returning `(w, b)`.

In [ ]:
def logreg_loss_grads(w, b, X, y):
    # TODO (dz = (p - y) / N)
    raise NotImplementedError

def train_logreg(X, y, lr=0.5, steps=500):
    w, b = np.zeros(X.shape[1]), 0.0
    # TODO
    raise NotImplementedError

In [ ]:
w, b = train_logreg(Xb, yb)
acc_blobs = np.mean((sigmoid(Xb @ w + b) > 0.5) == yb)
w2, b2 = train_logreg(Xs, ys)
acc_spirals = np.mean((sigmoid(Xs @ w2 + b2) > 0.5) == ys)
print(f"blobs accuracy {acc_blobs:.2f}, spirals accuracy {acc_spirals:.2f}")
assert acc_blobs > 0.9, "should separate the blobs"
assert acc_spirals < 0.8, "a straight line can't separate spirals"
plot_boundary(lambda P: sigmoid(P @ w2 + b2) > .5, Xs, ys, "logistic regression on spirals")
ok("logistic regression (and its limit)")

## Exercise 2: the MLP forward and backward pass
Same as week 2, now for training. `init_params(d, h, C)` uses weights ~ N(0, 1)/sqrt(fan_in) and zero biases. `forward` returns `(H, P)`; `backward` returns the 4 gradients.

In [ ]:
def init_params(d, h, C, seed=0):
    # TODO
    raise NotImplementedError

def forward(params, X):
    # TODO: H = tanh(X W1 + b1); P = softmax(H W2 + b2) (stable)
    raise NotImplementedError

def backward(params, X, y, H, P):
    # TODO
    raise NotImplementedError

In [ ]:
p = init_params(2, 8, 2)
H, P = forward(p, Xs[:16]); g = backward(p, Xs[:16], ys[:16], H, P)
assert [x.shape for x in p] == [(2, 8), (8,), (8, 2), (2,)] and [x.shape for x in g] == [x.shape for x in p]
assert np.allclose(P.sum(1), 1)
def loss(): return -np.mean(np.log(forward(p, Xs[:16])[1][np.arange(16), ys[:16]]))
i = (0, 3); old = p[0][i]; p[0][i] = old + 1e-5; Lp = loss(); p[0][i] = old - 1e-5; Lm = loss(); p[0][i] = old
assert np.isclose((Lp - Lm) / 2e-5, g[0][i], rtol=1e-4, atol=1e-9)
ok("forward/backward")

## Exercise 3: Adam
Write `adam_step(params, grads, state, lr, b1=0.9, b2=0.999, eps=1e-8)`. `state` is a dict you create on the first call with `m`, `v` (lists of zero arrays) and `t = 0`. Update in place, with bias correction, exactly as in the lesson.

In [ ]:
def adam_step(params, grads, state, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
    if not state:
        state.update(m=[np.zeros_like(p) for p in params], v=[np.zeros_like(p) for p in params], t=0)
    # TODO: t += 1; for each param: update m, v, compute m_hat, v_hat, p -= lr * m_hat / (sqrt(v_hat) + eps)
    raise NotImplementedError

In [ ]:
# minimise f(x) = (x - 3)^2 with Adam
x = [np.array([0.0])]; st = {}
for _ in range(500): adam_step(x, [2 * (x[0] - 3)], st, lr=0.1)
assert abs(x[0][0] - 3) < 1e-2, x
# first step size is ~lr regardless of gradient scale (bias correction)
y_ = [np.array([0.0])]; st2 = {}; adam_step(y_, [np.array([1e-6])], st2, lr=0.1)
assert np.isclose(y_[0][0], -0.1, atol=1e-3)
ok("Adam")

## Exercise 4: train on the spirals, SGD vs Adam
Write `train(optimizer, hidden=64, steps=3000, lr=...)` that trains on the full spiral set and returns `(params, losses)`. For SGD use `p -= lr * g` (try lr≈1.0); for Adam use your `adam_step` (lr≈0.02). Reach at least 95% training accuracy with one of them.

In [ ]:
def train(optimizer="adam", hidden=64, steps=3000, lr=None, seed=0):
    # TODO
    raise NotImplementedError

In [ ]:
p_sgd, l_sgd = train("sgd"); p_adam, l_adam = train("adam")
print(f"SGD acc {accuracy(p_sgd, Xs, ys):.3f} | Adam acc {accuracy(p_adam, Xs, ys):.3f}")
assert max(accuracy(p_sgd, Xs, ys), accuracy(p_adam, Xs, ys)) > 0.95
plt.plot(l_sgd, label="SGD"); plt.plot(l_adam, label="Adam"); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()
plot_boundary(lambda P: forward(p_adam, P)[1].argmax(1), Xs, ys, "MLP + Adam")
ok("spirals solved")

## Explore (no check)
Try `hidden = 2, 8, 64`. What is the smallest hidden size that solves the spirals? Note it in your weekly journal; the lesson asks for it.

In [ ]:
for h in [2, 8, 64]:
    p_, _ = train("adam", hidden=h, steps=3000)
    print(h, "hidden units ->", round(accuracy(p_, Xs, ys), 3))

---
## Solutions

Try every exercise first. Run the cell below only to compare with your own version; it overwrites your functions, so re-run the Check cells afterwards.

In [ ]:
def logreg_loss_grads(w, b, X, y):
    p = sigmoid(X @ w + b); eps = 1e-12
    L = -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))
    dz = (p - y) / len(y)
    return L, X.T @ dz, dz.sum()

def train_logreg(X, y, lr=0.5, steps=500):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(steps):
        _, dw, db = logreg_loss_grads(w, b, X, y); w -= lr * dw; b -= lr * db
    return w, b

def init_params(d, h, C, seed=0):
    r = np.random.default_rng(seed)
    return [r.normal(size=(d, h)) / np.sqrt(d), np.zeros(h), r.normal(size=(h, C)) / np.sqrt(h), np.zeros(C)]

def forward(params, X):
    W1, b1, W2, b2 = params
    H = np.tanh(X @ W1 + b1); Z = H @ W2 + b2
    Z = Z - Z.max(1, keepdims=True); P = np.exp(Z); P /= P.sum(1, keepdims=True)
    return H, P

def backward(params, X, y, H, P):
    W1, b1, W2, b2 = params; N = len(y)
    dZ = P.copy(); dZ[np.arange(N), y] -= 1; dZ /= N
    dA = (dZ @ W2.T) * (1 - H ** 2)
    return [X.T @ dA, dA.sum(0), H.T @ dZ, dZ.sum(0)]

def adam_step(params, grads, state, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
    if not state:
        state.update(m=[np.zeros_like(p) for p in params], v=[np.zeros_like(p) for p in params], t=0)
    state["t"] += 1; t = state["t"]
    for p, g, m, v in zip(params, grads, state["m"], state["v"]):
        m *= b1; m += (1 - b1) * g
        v *= b2; v += (1 - b2) * g * g
        p -= lr * (m / (1 - b1 ** t)) / (np.sqrt(v / (1 - b2 ** t)) + eps)

def train(optimizer="adam", hidden=64, steps=3000, lr=None, seed=0):
    params = init_params(2, hidden, 2, seed); state = {}; losses = []
    lr = lr or (0.02 if optimizer == "adam" else 1.0)
    for _ in range(steps):
        H, P = forward(params, Xs)
        losses.append(-np.mean(np.log(P[np.arange(len(ys)), ys] + 1e-12)))
        g = backward(params, Xs, ys, H, P)
        if optimizer == "adam":
            adam_step(params, g, state, lr)
        else:
            for p, gg in zip(params, g): p -= lr * gg
    return params, losses